## 7. Méthode du recuit simulé
### 7.1 Définition des fonctions et test sur l'optimal

On partira de la solution optimale obtenue précédemment pour tester le recuit simulé. 

In [1]:
import Pkg
Pkg.activate("..")

using JuMP
using HiGHS
using Random
using Statistics
include("../src/modele_v3.jl")
include("../src/donnees.jl")
include("../src/recuit.jl")

Random.seed!(1337)

data = charger_donnees();

  Activating project at `C:\Dev\workforce-scheduling-optimization`


In [2]:
#On récupère la solution optimale
function resoudre(data; p_min = 1, p_semaine = 1)
    model, x, e, viol_type, viol_jour, viol_semaine =
        construire_modele(data; p_min, p_semaine)
    set_silent(model)
    optimize!(model)
    return (; model, x, e, viol_type, viol_jour, viol_semaine)
end

model, x, e, viol_type, viol_jour, viol_semaine = resoudre(data; p_min = 1, p_semaine = 1);

In [3]:
print("Le coût du planning est de : ", calcul_cout(x,data)[1])

Le coût du planning est de : 18

In [4]:
#On teste la fonction en partant de la solution optimale trouvée avec HiGHS
x_opt, cout_opt = recuit_simule(x, data, 10.0, 0.95, 1000, 50)
print("Le meilleur coût est de ", cout_opt)

Le meilleur coût est de 18

Le meilleur planning est bien conservé, on a le coût minimal trouvé avec la programmation linéaire. 
On cherche un nouveau planning valide qui n'est pas celui optimal et on calculera son coût. 

In [5]:
Random.seed!(1337)

x_courant = round.(Int, Array(value.(x)))
liste = Matrix{Int}[]

for essai in 1:10_000
    x_temp, _, _ = voisin_valide(x_courant, data)

    if x_temp !== nothing
        push!(liste, x_temp)
        x_courant = x_temp
    end

    length(liste) == 30 && break
end

println("Plannings générés : ", length(liste))
println("Coût du dernier : ", calcul_cout(liste[end], data)[1])

Plannings générés : 30
Coût du dernier : 51


On teste la fonction recuit_simule en partant d'un autre planning valide qu'on vient de générer (coût : 51).

In [6]:
x1, c1, nb_valide, nb_genere_total, nb_invalide, nb_accepte, nb_ameliore, nb_degrade = recuit_simule(liste[30], data, 10.0, 0.95, 1000, 50)

([0 0 … 0 0; 1 0 … 0 0; … ; 0 0 … 0 0; 0 0 … 0 0], 22, 42824, 444823, 401999, 32774, 4717, 4758)

On a bien amélioré le résultat avec la méthode du recuit simulé (on est passé d'un coût de 51 à 22). Le planning est valide car il est issu de la fonction voisin_valide qui appelle controleContraintes.



In [7]:
println("Voisins construits : ", nb_genere_total )
println("Voisins invalides : ", nb_invalide)
println("Voisins valides : ", nb_valide)

Voisins construits : 444823
Voisins invalides : 401999
Voisins valides : 42824


In [8]:
println("Voisins acceptés : ", nb_accepte)
println("Voisins améliorés : ", nb_ameliore)
println("Voisins dégradés : ", nb_degrade)

Voisins acceptés : 32774
Voisins améliorés : 4717
Voisins dégradés : 4758


90% des plannings générés sont invalides. Mais on a généré 50 000 appels (50 x 1000) et on a trouvé 42 824 voisins valides. Dans plus de 85% des cas, on trouve un voisin valide. 

Sur les 32 774 voisins acceptés :
* 4 717 améliorent la solution ;
* 4 758 dégradent la solution ;
* les autres (23 299) ont le même coût que la solution courante.

Le problème est maintenant lié au fait que les plannings sélectionnés se font sur la base d'une permutation et le nombre de shifts de chacun est maintenu. Il faut de nouvelles manières de choisir le voisinage et mettre à jour la fonction. 

In [9]:
x2, c2, nb_valide2, nb_genere_total2, nb_invalide2, nb_accepte2, nb_ameliore2, nb_degrade2 = recuit_simule2(liste[30], data, 10.0, 0.95, 1000, 50)

([0 0 … 0 0; 1 0 … 0 0; … ; 0 0 … 0 1; 0 1 … 0 0], 30, 49976, 216424, 166448, 35294, 12341, 12287)

In [10]:
println("Voisins construits : ", nb_genere_total2 )
println("Voisins invalides : ", nb_invalide2)
println("Voisins valides : ", nb_valide2)
println("Voisins acceptés : ", nb_accepte2)
println("Voisins améliorés : ", nb_ameliore2)
println("Voisins dégradés : ", nb_degrade2)

Voisins construits : 216424
Voisins invalides : 166448
Voisins valides : 49976
Voisins acceptés : 35294
Voisins améliorés : 12341
Voisins dégradés : 12287


On a maintenant 77% d'invalides, c'est moins qu'avant. Et dans presque 100% des cas, on a trouvé un voisin valide. On a presque autant d'améliorations que de dégradations comme avant. Par contre le coût minimum est de 30 alors qu'on avait trouvé 22 auparavant et l'optimalité est à 18. 

In [11]:
println("Pour le résultat du recuit simulé")

c, n1, n2, n3, n4 = calcul_cout(x2, data)
println("Coût : ", c)
println("Pénalités type de shift : ",n1)
println("Pénalités type de jour : ",n2)
println("Pénalités shifts minimum : ",n3)
println("Pénalités semaines chargées : ",n4)

Pour le résultat du recuit simulé
Coût : 30
Pénalités type de shift : 9
Pénalités type de jour : 11
Pénalités shifts minimum : 7
Pénalités semaines chargées : 3


In [12]:
println("Pour le résultat de la programmation linéaire")

c, n1, n2, n3, n4 = calcul_cout(x, data)
println("Coût :", c)
println("Pénalités type de shift : ",n1)
println("Pénalités type de jour : ",n2)
println("Pénalités shifts minimum : ",n3)
println("Pénalités semaines chargées : ",n4)

Pour le résultat de la programmation linéaire
Coût :18
Pénalités type de shift : 3
Pénalités type de jour : 9
Pénalités shifts minimum : 4
Pénalités semaines chargées : 2


### 7.2 Tests pour évaluer l'impact des paramètres 
On fait d'autres tests en modifiant les paramètres et en multipliant les essais pour voir la moyenne des coûts obtenus par set de paramètres. 

**Paramètres initiaux**

In [13]:
Random.seed!(1337)
set1 = []
for i in 1:10
    x1, cout1 = recuit_simule(liste[30], data, 10.0, 0.95, 1000, 50)
    push!(set1, cout1)
end

In [14]:
print("Moyenne du set1 : ", mean(set1)," \n")
print("Meilleur coût : ", minimum(set1), " \n")

Moyenne du set1 : 22.3 
Meilleur coût : 20 


**Paramètres modifiés : plus de paliers**

In [15]:
Random.seed!(1337)
set2 = []
for i in 1:10
    x2, cout2 = recuit_simule(liste[30], data, 10.0, 0.95, 1000, 90)
    push!(set2, cout2)
end

In [16]:
print("Moyenne du set2 : ", mean(set2)," \n")
print("Meilleur coût : ", minimum(set2), " \n")

Moyenne du set2 : 18.0 
Meilleur coût : 18 


On teste sur un planning trouvé par l'algorithme glouton. 

In [17]:
Random.seed!(1337)
x_depart, nb_essais = construire_planning(data)

if x_depart === nothing
    println("Aucun planning valide trouvé")
else
    println("Essais nécessaires : ", nb_essais)
    println("Coût de départ : ", calcul_cout(x_depart, data)[1])
    println("Valide : ", controleContraintes(x_depart, data))
end

Essais nécessaires : 1
Coût de départ : 114
Valide : true


In [18]:
Random.seed!(1337)
set3 = []
x3 = nothing
for i in 1:10
    x3, cout3 = recuit_simule(x_depart, data, 10.0, 0.95, 1000, 90)
    push!(set3, cout3)
end

In [19]:
print("Moyenne du set3 : ", mean(set3)," \n")
print("Meilleur coût : ", minimum(set3), " \n")

Moyenne du set3 : 37.0 
Meilleur coût : 37 


In [20]:
d = calcul_cout(x_depart, data)
r = calcul_cout(x3, data)

println("Coût : ", d[1], " → ", r[1])
println("Préférences de type : ", d[2], " → ", r[2])
println("Préférences de jour : ", d[3], " → ", r[3])
println("Shifts minimum : ", d[4], " → ", r[4])
println("Semaines chargées : ", d[5], " → ", r[5])
println("Totaux par employé identiques : ", sum(x_depart, dims = 2) == sum(x3, dims = 2))

Coût : 114 → 37
Préférences de type : 79 → 3
Préférences de jour : 10 → 9
Shifts minimum : 12 → 12
Semaines chargées : 13 → 13
Totaux par employé identiques : true


Sur la version optimale trouvée par HiGHS, on a trouvé précédemment :
* Pénalités type de shift : 3
* Pénalités type de jour : 9
* Pénalités shifts minimum : 4
* Pénalités semaines chargées : 2

Dans ce planning optimisé, on a :
* Pénalités type de shift : 3
* Pénalités type de jour : 9
* Pénalités shifts minimum : 12 (identique au résultat de l'algorithme glouton)
* Pénalités semaines chargées : 13 (identique au résultat de l'algorithme glouton)

On n'a pas d'amélioration sur les shifts minimums et les semaines chargées à cause de la fonction de recherche de voisinnage. On va appliquer la fonction recuit_simule2 pour trouver les voisins autrement et on contrôle le temps de calcul.

In [21]:
Random.seed!(1337)

couts = Int[]
temps = Float64[]

recuit_simule2(x_depart, data, 10.0, 0.95, 1, 1)  # compilation, résultat ignoré
x4 = nothing
for i in 1:10
    t = @elapsed begin
        x4, cout4 = recuit_simule2(x_depart, data, 10.0, 0.95, 1000, 90)
    end
    push!(couts, cout4)
    push!(temps, t)
end

print("Moyenne du set : ", mean(couts)," \n")
print("Meilleur coût : ", minimum(couts), " \n")
println("Pire coût : ", maximum(couts))
println("Temps moyen : ", round(mean(temps), digits = 2), " s")
println("Temps médian : ", round(median(temps), digits = 2), " s")
println("Temps maximal : ", round(maximum(temps), digits = 2), " s")

Moyenne du set : 18.0 
Meilleur coût : 18 
Pire coût : 18
Temps moyen : 86.81 s
Temps médian : 86.84 s
Temps maximal : 87.56 s


In [22]:
# On teste si le planning trouvé est bien valide
m_test, x_test, = construire_modele(data; p_min = 1, p_semaine = 1)

for (a, i) in enumerate(data.I), (b, j) in enumerate(data.J)
    fix(x_test[i, j], x4[a, b]; force = true)
end

set_silent(m_test)
optimize!(m_test)
println(termination_status(m_test))
is_solved_and_feasible(m_test) && println(objective_value(m_test))

OPTIMAL
18.0


Le planning trouvé est optimal et valide. On va chercher à réduire le nombre d'itérations maintenant.

In [23]:
for n_iter in [750,600,500]
    Random.seed!(1337)
    couts = Int[]
    temps = Float64[]

    for essai in 1:10
        t = @elapsed begin
            _, cout = recuit_simule2(x_depart, data, 10.0, 0.95, n_iter, 90)
        end
        push!(couts, cout)
        push!(temps, t)
    end

    println("n_iter = ", n_iter,
            " | atteint 18 : ", count(==(18), couts), "/10",
            " | moyenne : ", mean(couts),
            " | pire : ", maximum(couts),
            " | temps moyen : ", round(mean(temps), digits = 2), " s")
end

n_iter = 750 | atteint 18 : 10/10 | moyenne : 18.0 | pire : 18 | temps moyen : 60.84 s
n_iter = 600 | atteint 18 : 10/10 | moyenne : 18.0 | pire : 18 | temps moyen : 69.32 s
n_iter = 500 | atteint 18 : 9/10 | moyenne : 18.1 | pire : 19 | temps moyen : 40.34 s


On vérifie sur un planning généré par l'algorithme glouton si on arrive bien à un planning optimal avec 600 itérations. 

In [24]:
Random.seed!(1337)

couts = Int[]
temps = Float64[]

recuit_simule2(x_depart, data, 10.0, 0.95, 1, 1)  # compilation, résultat ignoré

for i in 1:30
    t = @elapsed begin
        x, cout = recuit_simule2(x_depart, data, 10.0, 0.95, 600, 90)
    end
    push!(couts, cout)
    push!(temps, t)
end

print("Moyenne du set : ", mean(couts)," \n")
print("Meilleur coût : ", minimum(couts), " \n")
println("Pire coût : ", maximum(couts))
println("Temps moyen : ", round(mean(temps), digits = 2), " s")
println("Temps médian : ", round(median(temps), digits = 2), " s")
println("Temps maximal : ", round(maximum(temps), digits = 2), " s")

Moyenne du set : 18.0 
Meilleur coût : 18 
Pire coût : 18
Temps moyen : 1416.86 s
Temps médian : 43.69 s
Temps maximal : 38296.38 s


On teste sur 5 nouveaux plannings générés par l'algorithme glouton. 

In [25]:
Random.seed!(2026)
departs = Matrix{Int}[]

while length(departs) < 5
    x_temp, nb_essais = construire_planning(data)

    if x_temp === nothing
        error("Aucun planning valide trouvé")
    end

    if !(x_temp in departs)
        push!(departs, x_temp)
        println("Départ ", length(departs),
                " | coût : ", calcul_cout(x_temp, data)[1],
                " | essais gloutons : ", nb_essais)
    end
end

Départ 1 | coût : 94 | essais gloutons : 1
Départ 2 | coût : 109 | essais gloutons : 2
Départ 3 | coût : 130 | essais gloutons : 1
Départ 4 | coût : 92 | essais gloutons : 2
Départ 5 | coût : 106 | essais gloutons : 1


In [26]:
for (d, x_d) in enumerate(departs)
    Random.seed!(1337 + d)
    couts = Int[]
    temps = Float64[]

    for essai in 1:10
        t = @elapsed begin
            _, cout = recuit_simule2(x_d, data, 10.0, 0.95, 600, 90)
        end
        push!(couts, cout)
        push!(temps, t)
    end

    println("Départ ", d,
            " | atteint 18 : ", count(==(18), couts), "/10",
            " | moyenne : ", mean(couts),
            " | pire : ", maximum(couts),
            " | temps moyen : ", round(mean(temps), digits = 2), " s")
end

Départ 1 | atteint 18 : 10/10 | moyenne : 18.0 | pire : 18 | temps moyen : 49.06 s
Départ 2 | atteint 18 : 10/10 | moyenne : 18.0 | pire : 18 | temps moyen : 50.92 s
Départ 3 | atteint 18 : 10/10 | moyenne : 18.0 | pire : 18 | temps moyen : 48.85 s
Départ 4 | atteint 18 : 10/10 | moyenne : 18.0 | pire : 18 | temps moyen : 48.42 s
Départ 5 | atteint 18 : 9/10 | moyenne : 18.1 | pire : 19 | temps moyen : 45.18 s


Avec 54 000 itérations, le recuit atteint le coût optimal de 18 dans 49 essais sur 50, depuis 5 plannings gloutons différents. Le seul échec atteint 19. On conserve les 600 itérations.

In [27]:
model, x, e, viol_type, viol_jour, viol_semaine = resoudre(data; p_min = 1, p_semaine = 1)

println("Statut : ", termination_status(model))
println("Coût : ", objective_value(model))
println("Temps HiGHS : ", round(solve_time(model), digits = 2), " s")

Statut : OPTIMAL
Coût : 18.0
Temps HiGHS : 0.03 s


HiGHS est nettement plus rapide avec ces données initiales pour trouver une solution : 0,02s contre 48s environ. On va donc tester les différences avec des données plus importantes dans le prochain notebook : plus d'employés et plus de shifts. On va aussi régler la question du respect du droit (pas de journées de 16h ni semaines qui vont jusqu'à 80h). 